# Diabetic Retinopathy Classification - No Transformations Version

Multi-class CNN classifier for diabetic retinopathy stages (0–4).

## Key Differences from Main Notebook

This notebook is a **baseline version** that trains on raw images **without any preprocessing or augmentation**:

### What's Removed
- ❌ No retinal enhancement (disk cropping, CLAHE, vessel masks)
- ❌ No data augmentation (no flips, rotations, color jitter, gaussian blur, random erasing)
- ❌ No random cropping

### What's Used
- ✅ Raw fundus images from `data/` folder
- ✅ Simple resize to 320×320
- ✅ ImageNet normalization only
- ✅ **In-memory dataset loading** for faster training (all images pre-loaded into RAM)

### Purpose
This notebook serves as a **baseline** to compare against the full preprocessing pipeline. It answers the question: "How much do transformations actually help?"

---

**Dataset**: `data/` (1216 images, 5 classes) — random 70/15/15 stratified split

**Architecture**: EfficientNetB0 (pretrained ImageNet) with custom classifier head

Classes: `0-noDR` | `1-mild` | `2-moderate` | `3-severe` | `4-proliferativeDR`

In [8]:
import os
# os.chdir('/Users/edu/Documents/Universidad/Master_Ingeniería_Informática/clases/iaiao/p3')
print(f'Working directory: {os.getcwd()}')

Working directory: /Users/edu/Documents/Universidad/Master_Ingeniería_Informática/clases/iaiao/p3


In [9]:
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from torchvision.models import EfficientNet_B0_Weights

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import random, os, time
import numpy as np
from PIL import Image, ImageOps
from collections import Counter

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)

print(f'Using device: {device}')

DATA_DIR  = 'data'
model_dir = 'models'
os.makedirs(model_dir, exist_ok=True)

CLASS_NAMES   = ['0-noDR', '1-mild', '2-moderate', '3-severe', '4-proliferativeDR']
NUM_CLASSES   = len(CLASS_NAMES)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
print(f'Classes ({NUM_CLASSES}): {CLASS_NAMES}')

Using device: mps
Classes (5): ['0-noDR', '1-mild', '2-moderate', '3-severe', '4-proliferativeDR']


## Dataset - In-Memory Loading

The `data/` directory contains **1216 raw fundus images** in 5 class folders (not pre-split).
We apply a random 70/15/15 stratified split into train/val/test.

**Key optimization**: All images are **pre-loaded into memory** during dataset initialization. This eliminates disk I/O during training, significantly speeding up each epoch.

| Class | Count |
|---|---|
| 0-noDR | 467 |
| 1-mild | 213 |
| 2-moderate | 354 |
| 3-severe | 95 |
| 4-proliferativeDR | 87 |
| **Total** | **1216** |

The dataset is **highly imbalanced**. We use inverse-frequency class weights in the loss function to handle this.

**Memory usage**: ~5-6 GB for all images as tensors (320×320×3 float32)

In [10]:
class FlatImageDataset(Dataset):
    """
    Loads images from root/class_name/image.jpg flat-folder structure.
    Supports an optional index subset for train/val/test splitting.
    """
    def __init__(self, root_dir, indices=None, transform=None):
        self.transform = transform
        self.image_paths = []
        self.labels = []
        class_names = sorted([d for d in os.listdir(root_dir)
                               if os.path.isdir(os.path.join(root_dir, d))])
        for class_name in class_names:
            class_dir = os.path.join(root_dir, class_name)
            label = int(class_name.split('-')[0])
            for fname in sorted(os.listdir(class_dir)):
                # Skip macOS system files
                if fname.startswith('._'):
                    continue
                if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.image_paths.append(os.path.join(class_dir, fname))
                    self.labels.append(label)
        if indices is not None:
            self.image_paths = [self.image_paths[i] for i in indices]
            self.labels      = [self.labels[i]      for i in indices]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        with Image.open(self.image_paths[idx]) as img:
            img = img.convert('RGB')
            if self.transform:
                img = self.transform(img)
        return img, self.labels[idx]


class InMemoryImageDataset(Dataset):
    """
    Pre-loads all images into memory for faster training.
    Loads images once during initialization, then returns cached tensors.
    """
    def __init__(self, root_dir, indices=None, transform=None, augmentation_transform=None):
        """
        Args:
            root_dir: root directory with class subdirectories
            indices: optional list of indices to use (for train/val/test split)
            transform: base transform to apply once during loading (resize, normalize)
            augmentation_transform: additional transforms to apply on-the-fly (flips, rotations, etc.)
        """
        print(f'Loading images into memory from {root_dir}...')
        self.augmentation_transform = augmentation_transform
        
        # Collect image paths and labels
        image_paths = []
        labels = []
        class_names = sorted([d for d in os.listdir(root_dir)
                               if os.path.isdir(os.path.join(root_dir, d))])
        for class_name in class_names:
            class_dir = os.path.join(root_dir, class_name)
            label = int(class_name.split('-')[0])
            for fname in sorted(os.listdir(class_dir)):
                # Skip macOS system files
                if fname.startswith('._'):
                    continue
                if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                    image_paths.append(os.path.join(class_dir, fname))
                    labels.append(label)
        
        # Apply index subset if provided
        if indices is not None:
            image_paths = [image_paths[i] for i in indices]
            labels = [labels[i] for i in indices]
        
        # Pre-load all images into memory
        self.images = []
        self.labels = labels
        start_time = time.time()
        
        for idx, path in enumerate(image_paths):
            with Image.open(path) as img:
                img = img.convert('RGB')
                if transform:
                    img = transform(img)
                self.images.append(img)
            
            if (idx + 1) % 200 == 0:
                elapsed = time.time() - start_time
                print(f'  Loaded {idx+1}/{len(image_paths)} images ({elapsed:.1f}s)')
        
        elapsed = time.time() - start_time
        print(f'  Loaded all {len(image_paths)} images in {elapsed:.1f}s')
        
        # Calculate memory usage estimate
        if len(self.images) > 0 and torch.is_tensor(self.images[0]):
            mem_per_image = self.images[0].element_size() * self.images[0].nelement()
            total_mem_mb = (mem_per_image * len(self.images)) / (1024 * 1024)
            print(f'  Estimated memory usage: {total_mem_mb:.1f} MB')

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        label = self.labels[idx]
        
        # Apply augmentation transforms on-the-fly if provided
        if self.augmentation_transform is not None:
            img = self.augmentation_transform(img)
        
        return img, label


def stratified_split(root_dir, train_frac=0.70, val_frac=0.15, seed=42):
    """Return (train_indices, val_indices, test_indices) via per-class stratified split."""
    rng = random.Random(seed)
    class_names = sorted([d for d in os.listdir(root_dir)
                           if os.path.isdir(os.path.join(root_dir, d))])
    all_labels = []
    idx_counter = 0
    class_to_indices = {}
    for class_name in class_names:
        class_dir = os.path.join(root_dir, class_name)
        label = int(class_name.split('-')[0])
        idxs = []
        for fname in sorted(os.listdir(class_dir)):
            # Skip macOS system files
            if fname.startswith('._'):
                continue
            if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                idxs.append(idx_counter)
                all_labels.append(label)
                idx_counter += 1
        class_to_indices[label] = idxs

    train_idx, val_idx, test_idx = [], [], []
    for label, idxs in class_to_indices.items():
        shuffled = idxs[:]
        rng.shuffle(shuffled)
        n = len(shuffled)
        n_train = max(1, int(n * train_frac))
        n_val   = max(1, int(n * val_frac))
        train_idx.extend(shuffled[:n_train])
        val_idx.extend(  shuffled[n_train:n_train + n_val])
        test_idx.extend( shuffled[n_train + n_val:])

    return train_idx, val_idx, test_idx


# Base transforms: just resize and normalize (applied once during loading)
base_transform = transforms.Compose([
    transforms.Resize((320, 320),
                      interpolation=transforms.InterpolationMode.LANCZOS),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# For training, we don't use augmentation in the no-transformation version
# For validation/test, no additional transforms needed
train_aug_transform = None
val_aug_transform = None

# Stratified 70/15/15 split
train_idx, val_idx, test_idx = stratified_split(DATA_DIR, train_frac=0.70, val_frac=0.15, seed=SEED)

# Load all images into memory
train_dataset = InMemoryImageDataset(DATA_DIR, indices=train_idx, 
                                     transform=base_transform,
                                     augmentation_transform=train_aug_transform)
val_dataset = InMemoryImageDataset(DATA_DIR, indices=val_idx, 
                                   transform=base_transform,
                                   augmentation_transform=val_aug_transform)
test_dataset = InMemoryImageDataset(DATA_DIR, indices=test_idx, 
                                    transform=base_transform,
                                    augmentation_transform=val_aug_transform)

train_counts = Counter(train_dataset.labels)
val_counts   = Counter(val_dataset.labels)
test_counts  = Counter(test_dataset.labels)

print(f'\nDataset statistics:')
print(f'Train: {len(train_dataset)} images')
for name in CLASS_NAMES:
    c = int(name.split('-')[0])
    print(f'  {name}: {train_counts[c]}')
print(f'Val:   {len(val_dataset)} images')
for name in CLASS_NAMES:
    c = int(name.split('-')[0])
    print(f'  {name}: {val_counts[c]}')
print(f'Test:  {len(test_dataset)} images')
for name in CLASS_NAMES:
    c = int(name.split('-')[0])
    print(f'  {name}: {test_counts[c]}')

Loading images into memory from data...
  Loaded 200/848 images (9.3s)
  Loaded 400/848 images (19.2s)
  Loaded 600/848 images (28.8s)
  Loaded 800/848 images (38.8s)
  Loaded all 848 images in 41.3s
  Estimated memory usage: 993.8 MB
Loading images into memory from data...
  Loaded all 181 images in 8.2s
  Estimated memory usage: 212.1 MB
Loading images into memory from data...
  Loaded all 187 images in 9.3s
  Estimated memory usage: 219.1 MB

Dataset statistics:
Train: 848 images
  0-noDR: 326
  1-mild: 149
  2-moderate: 247
  3-severe: 66
  4-proliferativeDR: 60
Val:   181 images
  0-noDR: 70
  1-mild: 31
  2-moderate: 53
  3-severe: 14
  4-proliferativeDR: 13
Test:  187 images
  0-noDR: 71
  1-mild: 33
  2-moderate: 54
  3-severe: 15
  4-proliferativeDR: 14


## Class Weights

Inverse-frequency class weights are applied in the loss function to handle class imbalance.
No WeightedRandomSampler — experiments showed it hurts performance on this dataset.

In [11]:
label_counts = Counter(train_dataset.labels)
total = len(train_dataset.labels)
class_weights = torch.zeros(NUM_CLASSES)
for i in range(NUM_CLASSES):
    class_weights[i] = total / (NUM_CLASSES * label_counts.get(i, 1))

print('Class weights (inverse frequency):')
for i, (name, w) in enumerate(zip(CLASS_NAMES, class_weights)):
    print(f'  {name}: weight={w:.4f}  (n={label_counts[i]})')
class_weights = class_weights.to(device)

Class weights (inverse frequency):
  0-noDR: weight=0.5202  (n=326)
  1-mild: weight=1.1383  (n=149)
  2-moderate: weight=0.6866  (n=247)
  3-severe: weight=2.5697  (n=66)
  4-proliferativeDR: weight=2.8267  (n=60)


# Usando un modelo no preentrenado

# Usando un modelo preentrenado de EfficientNetB0 para clasificar la retinopatía diabética en cinco etapas (0-4).

## Model Architecture

**EfficientNetB0** pretrained on ImageNet, custom head: `Dropout(0.3) -> Linear(1280 -> 5)`

Best architecture for this dataset size — larger models overfit.

In [12]:
def build_model(num_classes=5, dropout=0.3, pretrained=True):
    """EfficientNetB0 with custom head for retinopathy classification."""
    weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
    model = models.efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, num_classes),
    )
    return model


model = build_model(num_classes=NUM_CLASSES, dropout=0.3, pretrained=True).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Architecture: EfficientNetB0 (pretrained ImageNet)')
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Classifier head:      Dropout(0.3) -> Linear(1280, {NUM_CLASSES})')

Architecture: EfficientNetB0 (pretrained ImageNet)
Total parameters:     4,013,953
Trainable parameters: 4,013,953
Classifier head:      Dropout(0.3) -> Linear(1280, 5)


## Training

Best hyperparameters:
- **Optimizer**: AdamW, lr=3e-4, weight_decay=1e-4
- **Scheduler**: OneCycleLR (max_lr=3e-4, pct_start=0.1, cosine annealing) — steps per batch
- **Loss**: CrossEntropyLoss + inverse-freq class weights + label_smoothing=0.1
- **Batch size**: 16, **Max epochs**: 200, **Early stopping patience**: 40
- **Gradient clipping**: max_norm=1.0
- **Image size**: 320x320 px
- **No augmentation or enhancement applied**

In [13]:
NUM_EPOCHS    = 200
BATCH_SIZE    = 16
LEARNING_RATE = 3e-4
WEIGHT_DECAY  = 1e-4
PATIENCE      = 40
BEST_MODEL_PATH = os.path.join(model_dir, 'best_model_effb0_320px.pth')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=False)
print(f'Train batches: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}')

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LEARNING_RATE,
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS,
    pct_start=0.1, anneal_strategy='cos',
)
print(f'Optimizer: AdamW  lr={LEARNING_RATE}  weight_decay={WEIGHT_DECAY}')
print(f'Scheduler: OneCycleLR  max_lr={LEARNING_RATE}')
print(f'Loss: CrossEntropyLoss(label_smoothing=0.1) + inverse-freq class weights')

Train batches: 53, Val: 12, Test: 12
Optimizer: AdamW  lr=0.0003  weight_decay=0.0001
Scheduler: OneCycleLR  max_lr=0.0003
Loss: CrossEntropyLoss(label_smoothing=0.1) + inverse-freq class weights


In [14]:
def evaluate(model, loader, criterion, device):
    """Evaluate model on loader. Returns avg loss and accuracy."""
    model.eval()
    total_loss = 0.0
    preds, true_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            total_loss += criterion(outputs, labels).item()
            preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), accuracy_score(true_labels, preds)


best_val_acc     = 0.0
patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

print(f'  Epoch  TrainLoss   ValLoss   ValAcc     Best')
print('-' * 52)

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    improved = val_acc > best_val_acc
    if improved:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
    else:
        patience_counter += 1

    if (epoch + 1) % 10 == 0 or epoch == 0 or improved:
        marker = ' *' if improved else ''
        print(f'  {epoch+1:>4}/{NUM_EPOCHS}  {avg_train_loss:>9.4f}  {val_loss:>8.4f}  {val_acc:>7.4f}  {best_val_acc:>7.4f}{marker}')

    if patience_counter >= PATIENCE:
        print(f'Early stopping at epoch {epoch+1}.')
        break

elapsed = time.time() - start_time
print(f'\nTraining finished in {elapsed/60:.1f} min')
print(f'Best Validation Accuracy: {best_val_acc:.4f}')

  Epoch  TrainLoss   ValLoss   ValAcc     Best
----------------------------------------------------
     1/200     1.7104    1.7690   0.2762   0.2762 *
     2/200     1.6718    1.7534   0.2983   0.2983 *
     6/200     1.3602    1.6262   0.3204   0.3204 *
     7/200     1.2829    1.5881   0.4199   0.4199 *
     9/200     1.0696    1.6072   0.4475   0.4475 *
    10/200     0.9786    1.5756   0.4254   0.4475
    11/200     0.8671    1.6327   0.4641   0.4641 *
    14/200     0.7226    1.6955   0.5138   0.5138 *
    20/200     0.6786    1.7001   0.5083   0.5138
    24/200     0.6181    1.8060   0.5193   0.5193 *
    30/200     0.6115    1.9494   0.4751   0.5193
    33/200     0.5927    1.8528   0.5304   0.5304 *
    40/200     0.5938    1.8491   0.5138   0.5304
    46/200     0.5753    1.7389   0.5470   0.5470 *
    50/200     0.5745    1.8362   0.4972   0.5470
    60/200     0.5713    1.9313   0.5470   0.5470
    63/200     0.5619    1.7485   0.5801   0.5801 *
    70/200     0.5639    2.1

## Evaluation on Test Set

In [15]:
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()

test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f'Test Loss:     {test_loss:.4f}')
print(f'Test Accuracy: {test_acc:.4f}')

all_preds, all_true = [], []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        all_preds.extend(torch.argmax(outputs, 1).cpu().numpy())
        all_true.extend(labels.numpy())

print('\nClassification Report:')
print(classification_report(all_true, all_preds, target_names=CLASS_NAMES, digits=4))
print('Confusion Matrix:')
print(confusion_matrix(all_true, all_preds))

Test Loss:     1.9367
Test Accuracy: 0.5294

Classification Report:
                   precision    recall  f1-score   support

           0-noDR     0.5591    0.7324    0.6341        71
           1-mild     0.0000    0.0000    0.0000        33
       2-moderate     0.4603    0.5370    0.4957        54
         3-severe     0.5714    0.5333    0.5517        15
4-proliferativeDR     0.7692    0.7143    0.7407        14

         accuracy                         0.5294       187
        macro avg     0.4720    0.5034    0.4845       187
     weighted avg     0.4486    0.5294    0.4836       187

Confusion Matrix:
[[52  4 15  0  0]
 [23  0 10  0  0]
 [18  0 29  5  2]
 [ 0  0  6  8  1]
 [ 0  0  3  1 10]]


## Second Model (for Ensemble)

Train a second EfficientNetB0 with a different random seed and earlier stopping point.
Ensembling two models trained on the same data but stopping at different times consistently
improves accuracy (+0.62% observed in R7+R8 ensemble on the full dataset).

In [16]:
BEST_MODEL2_PATH = os.path.join(model_dir, 'best_model2_effb0_320px.pth')
NUM_EPOCHS_2 = 120
PATIENCE_2   = 30

torch.manual_seed(SEED + 1)
random.seed(SEED + 1)
np.random.seed(SEED + 1)

model2 = build_model(num_classes=NUM_CLASSES, dropout=0.3, pretrained=True).to(device)

criterion2 = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer2 = optim.AdamW(model2.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler2 = optim.lr_scheduler.OneCycleLR(
    optimizer2, max_lr=LEARNING_RATE,
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS_2,
    pct_start=0.1, anneal_strategy='cos',
)

best_val_acc2     = 0.0
patience_counter2 = 0

print(f'Training second model (max {NUM_EPOCHS_2} epochs, patience {PATIENCE_2})...')
print(f'  Epoch  TrainLoss   ValLoss   ValAcc     Best')
print('-' * 52)

start2 = time.time()
for epoch in range(NUM_EPOCHS_2):
    model2.train()
    total_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer2.zero_grad()
        loss = criterion2(model2(images), labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model2.parameters(), max_norm=1.0)
        optimizer2.step()
        scheduler2.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    val_loss2, val_acc2 = evaluate(model2, val_loader, criterion2, device)

    improved2 = val_acc2 > best_val_acc2
    if improved2:
        best_val_acc2 = val_acc2
        patience_counter2 = 0
        torch.save(model2.state_dict(), BEST_MODEL2_PATH)
    else:
        patience_counter2 += 1

    if (epoch + 1) % 10 == 0 or epoch == 0 or improved2:
        marker = ' *' if improved2 else ''
        print(f'  {epoch+1:>4}/{NUM_EPOCHS_2}  {avg_train_loss:>9.4f}  {val_loss2:>8.4f}  {val_acc2:>7.4f}  {best_val_acc2:>7.4f}{marker}')

    if patience_counter2 >= PATIENCE_2:
        print(f'Early stopping at epoch {epoch+1}.')
        break

elapsed2 = time.time() - start2
print(f'\nTraining finished in {elapsed2/60:.1f} min')
print(f'Best Validation Accuracy (model2): {best_val_acc2:.4f}')

Training second model (max 120 epochs, patience 30)...
  Epoch  TrainLoss   ValLoss   ValAcc     Best
----------------------------------------------------
     1/120     1.7230    1.7954   0.1326   0.1326 *
     2/120     1.6600    1.7572   0.1768   0.1768 *
     3/120     1.5989    1.7208   0.2541   0.2541 *
     4/120     1.4661    1.6547   0.3149   0.3149 *
     5/120     1.3411    1.6162   0.3646   0.3646 *
     6/120     1.1895    1.5487   0.3978   0.3978 *
     7/120     1.0726    1.5886   0.4144   0.4144 *
     9/120     0.8839    1.6936   0.4199   0.4199 *
    10/120     0.7930    1.6743   0.5083   0.5083 *
    14/120     0.7133    1.7345   0.5193   0.5193 *
    19/120     0.6577    1.6963   0.5304   0.5304 *
    20/120     0.6490    1.6838   0.5028   0.5304
    29/120     0.5837    1.7313   0.5470   0.5470 *
    30/120     0.5837    1.8391   0.4917   0.5470
    32/120     0.5855    1.6969   0.5691   0.5691 *
    40/120     0.5784    1.7191   0.5083   0.5691
    50/120     0.56

## Ensemble (Best Overall)

Average softmax probabilities of the two EfficientNetB0 models.
- Model 1: up to 200 epochs (best val checkpoint)
- Model 2: up to 120 epochs (best val checkpoint)

This strategy consistently yields the best test accuracy.

In [17]:
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model2.load_state_dict(torch.load(BEST_MODEL2_PATH, map_location=device))
model.eval()
model2.eval()

ensemble_models = [model, model2]

# Individual model accuracies
_, acc1 = evaluate(model,  test_loader, criterion,  device)
_, acc2 = evaluate(model2, test_loader, criterion2, device)
print(f'Model 1 (200ep) Test Accuracy: {acc1:.4f}')
print(f'Model 2 (120ep) Test Accuracy: {acc2:.4f}')

# Ensemble: average softmax probabilities
ens_preds, ens_true = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        prob_sum = sum(F.softmax(m(images), dim=1) for m in ensemble_models)
        ens_preds.extend(torch.argmax(prob_sum, 1).cpu().numpy())
        ens_true.extend(labels.numpy())

ens_acc = accuracy_score(ens_true, ens_preds)
print(f'\nEnsemble Test Accuracy: {ens_acc:.4f}')
print('\nEnsemble Classification Report:')
print(classification_report(ens_true, ens_preds, target_names=CLASS_NAMES, digits=4))
print('Confusion Matrix:')
print(confusion_matrix(ens_true, ens_preds))

Model 1 (200ep) Test Accuracy: 0.5294
Model 2 (120ep) Test Accuracy: 0.4813

Ensemble Test Accuracy: 0.4973

Ensemble Classification Report:
                   precision    recall  f1-score   support

           0-noDR     0.4952    0.7324    0.5909        71
           1-mild     0.3077    0.1212    0.1739        33
       2-moderate     0.4634    0.3519    0.4000        54
         3-severe     0.5000    0.5333    0.5161        15
4-proliferativeDR     0.8333    0.7143    0.7692        14

         accuracy                         0.4973       187
        macro avg     0.5199    0.4906    0.4900       187
     weighted avg     0.4786    0.4973    0.4695       187

Confusion Matrix:
[[52  8 11  0  0]
 [25  4  4  0  0]
 [26  1 19  7  1]
 [ 2  0  4  8  1]
 [ 0  0  3  1 10]]


## Sample Inference

Test the ensemble on random samples from the test set.

In [18]:
idx_to_class = {int(name.split('-')[0]): name for name in CLASS_NAMES}

print('Sample predictions (ensemble):')
print('-' * 65)
print(f"  {'Idx':>5}  {'True Label':>22}  {'Predicted':>22}  {'Correct':>7}")
print('-' * 65)

random.seed(0)
for _ in range(10):
    idx = random.randint(0, len(test_dataset) - 1)
    img, true_label = test_dataset[idx]
    with torch.no_grad():
        prob_sum = sum(F.softmax(m(img.unsqueeze(0).to(device)), dim=1) for m in ensemble_models)
        pred = torch.argmax(prob_sum, 1).item()
    correct = 'YES' if true_label == pred else 'NO'
    print(f"  {idx:>5}  {idx_to_class[true_label]:>22}  {idx_to_class[pred]:>22}  {correct:>7}")

Sample predictions (ensemble):
-----------------------------------------------------------------
    Idx              True Label               Predicted  Correct
-----------------------------------------------------------------
     98                  1-mild                  1-mild      YES
    107              2-moderate                  0-noDR       NO
     10                  0-noDR                  0-noDR      YES
     66                  0-noDR                  0-noDR      YES
    130              2-moderate              2-moderate      YES
    124              2-moderate              2-moderate      YES
    103                  1-mild                  0-noDR       NO
     77                  1-mild              2-moderate       NO
    122              2-moderate                  0-noDR       NO
     91                  1-mild                  0-noDR       NO
